# Experiment: Aspect-based Sentiment Analysis (ABSA)

Objective: approximate aspect-level sentiment by mining frequent aspect keywords
and aggregating predicted sentiment for reviews containing each aspect.


## 1. Setup


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.json"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.data import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


## 4. Data Validation


In [ ]:
assert train_df[LABEL_COL].isin([0, 1]).all()
assert train_df[REVIEW_COL].str.len().gt(0).all()


## 5. Exploratory Analysis


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words="english", min_df=10, ngram_range=(1, 1))
X = cv.fit_transform(train_df[REVIEW_COL])
term_counts = np.asarray(X.sum(axis=0)).ravel()
terms = np.array(cv.get_feature_names_out())

top = pd.DataFrame({"term": terms, "count": term_counts}).sort_values("count", ascending=False).head(100)
top.head(20)


## 6. Feature Engineering


In [ ]:
from sentiment_project.modeling import build_tfidf_logreg_pipeline

absa_model = build_tfidf_logreg_pipeline(random_state=RANDOM_SEED)
absa_model.fit(train_df[REVIEW_COL], train_df[LABEL_COL])

candidate_aspects = [
    t for t in top["term"].tolist() if len(t) > 3
][:20]
candidate_aspects[:10]


## 7. Modeling


In [ ]:
def aspect_sentiment_table(df: pd.DataFrame, aspect_terms: list[str]) -> pd.DataFrame:
    rows = []
    for aspect in aspect_terms:
        mask = df["reviews"].str.contains(rf"\b{aspect}\b", case=False, regex=True)
        subset = df.loc[mask, "reviews"]
        if len(subset) < 20:
            continue
        preds = absa_model.predict(subset)
        rows.append(
            {
                "aspect": aspect,
                "sample_count": int(len(subset)),
                "positive_rate": float(np.mean(preds == 1)),
                "negative_rate": float(np.mean(preds == 0)),
            }
        )
    return pd.DataFrame(rows).sort_values("sample_count", ascending=False)

aspect_df = aspect_sentiment_table(train_df, candidate_aspects)
aspect_df.head(15)


## 8. Evaluation


In [ ]:
ax = sns.barplot(data=aspect_df.head(12), x="aspect", y="positive_rate")
ax.set_title("Estimated positive sentiment by aspect")
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 9. Inference / Submission


In [ ]:
out = PROCESSED_DIR / "aspect_sentiment_summary.csv"
aspect_df.to_csv(out, index=False)
print(out.relative_to(PROJECT_ROOT))
aspect_df.head()


## 10. Conclusions / Next Steps

- This notebook provides weakly supervised ABSA from review-level labels.
- True ABSA should be trained with aspect-level annotations and token/span supervision.
- Current output is useful for coarse business insights by aspect keyword.
